In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load data ─────────────────────────────────────────────────────────────────
lsoas = pd.read_csv('data/test_lsoas.csv')
imd_full = pd.read_csv('data/govuk2.csv')

score_cols = [c for c in imd_full.columns if 'Score' in c]
lsoa_col = imd_full.columns[0]
imd_features = imd_full[[lsoa_col] + score_cols].copy()
imd_features.columns = ['LSOA11CD'] + [f'score_{i}' for i in range(len(score_cols))]
lsoas_merged = lsoas.merge(imd_features, on='LSOA11CD', how='inner')
score_feature_cols = [c for c in lsoas_merged.columns if c.startswith('score_')]

# Borough column for spatial CV
borough_col = [c for c in imd_full.columns if 'District' in c or 'Authority' in c or 'LAD' in c]
if borough_col:
    borough_map = imd_full[[lsoa_col, borough_col[0]]].copy()
    borough_map.columns = ['LSOA11CD', 'borough']
    lsoas_merged = lsoas_merged.merge(borough_map, on='LSOA11CD', how='left')
else:
    lsoas_merged['borough'] = lsoas_merged['LSOA11CD'].str[3:6]

print(f'LSOAs: {len(lsoas_merged)} | Boroughs: {lsoas_merged["borough"].nunique()}')

Device: cpu
LSOAs: 2000 | Boroughs: 54


In [2]:
# ── Build image records ───────────────────────────────────────────────────────
street_dir = 'data/images/'
valid_lsoas = set(lsoas_merged['LSOA11CD'].values)

records = []
for fname in os.listdir(street_dir):
    if not fname.endswith('.jpg') or fname == 'test_image.jpg': continue
    lsoa_code = fname.rsplit('_', 1)[0]
    if lsoa_code in valid_lsoas:
        row = lsoas_merged[lsoas_merged['LSOA11CD'] == lsoa_code].iloc[0]
        records.append({
            'path': os.path.join(street_dir, fname),
            'lsoa': lsoa_code,
            'imd': float(row['IMD19']),
            'borough': row['borough']
        })

img_df = pd.DataFrame(records)
print(f'Total street view images: {len(img_df)}')
print(f'LSOAs with images: {img_df["lsoa"].nunique()}')

Total street view images: 6924
LSOAs with images: 1731


In [3]:
# ── Dataset class ─────────────────────────────────────────────────────────────
class StreetViewDataset(Dataset):
    def __init__(self, df, transform, y_mean, y_std):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.y_mean = y_mean
        self.y_std = y_std

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        img = self.transform(img)
        y_norm = (row['imd'] - self.y_mean) / self.y_std
        return img, torch.tensor(y_norm, dtype=torch.float32)

# Augmentation for training
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# No augmentation for validation
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

print('Transforms defined')

Transforms defined


In [4]:
# ── Model: ResNet-50 with partial fine-tuning ─────────────────────────────────
class PartialFineTuneResNet(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        base = models.resnet50(weights='IMAGENET1K_V2')

        # Freeze everything first
        for p in base.parameters():
            p.requires_grad = False

        # Unfreeze layer3, layer4 (last two residual blocks)
        for p in base.layer3.parameters():
            p.requires_grad = True
        for p in base.layer4.parameters():
            p.requires_grad = True

        self.backbone = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4,
            base.avgpool
        )

        # Regression head with dropout
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

model = PartialFineTuneResNet(dropout=0.4).to(device)

# Count trainable params
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,}')
print(f'Trainable params: {trainable:,} ({100*trainable/total:.1f}%)')
print('Frozen: early layers (conv1, layer1, layer2)')
print('Trainable: layer3, layer4, regression head')

Total params: 24,032,833
Trainable params: 22,587,905 (94.0%)
Frozen: early layers (conv1, layer1, layer2)
Trainable: layer3, layer4, regression head


In [ ]:
# ── Spatial CV — Borough holdout ──────────────────────────────────────────────
unique_boroughs = img_df['borough'].unique()
np.random.seed(42)
shuffled = np.random.permutation(unique_boroughs)
borough_folds = np.array_split(shuffled, 5)

# Store OOF predictions at LSOA level
lsoa_list = sorted(img_df['lsoa'].unique())

lsoa_to_idx = {l: i for i, l in enumerate(lsoa_list)}
lsoa_imd = {row['lsoa']: row['imd'] for _, row in img_df.drop_duplicates('lsoa').iterrows()}

oof_lsoa_preds = {}  # lsoa -> list of image predictions
oof_lsoa_true  = {}

all_fold_r2 = []

for fold, test_boroughs in enumerate(borough_folds):
    print(f'\n=== Fold {fold+1}/5 | Test boroughs: {len(test_boroughs)} ===')

    train_df = img_df[~img_df['borough'].isin(test_boroughs)]
    val_df   = img_df[img_df['borough'].isin(test_boroughs)]

    print(f'Train images: {len(train_df)} | Val images: {len(val_df)}')

    # Normalise target on train
    y_mean = train_df['imd'].mean()
    y_std  = train_df['imd'].std()

    train_ds = StreetViewDataset(train_df, train_transform, y_mean, y_std)
    val_ds   = StreetViewDataset(val_df,   val_transform,   y_mean, y_std)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

    # Fresh model each fold
    fold_model = PartialFineTuneResNet(dropout=0.4).to(device)

    # Differential LR: smaller for pretrained layers, larger for head
    pretrained_params = list(fold_model.backbone.parameters())
    head_params       = list(fold_model.head.parameters())

    optimiser = optim.AdamW([
        {'params': pretrained_params, 'lr': 1e-5, 'weight_decay': 1e-4},
        {'params': head_params,       'lr': 1e-3, 'weight_decay': 1e-3}
    ])

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=20)
    criterion = nn.MSELoss()

    # Training loop with early stopping
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    best_state = None

    for epoch in range(30):
        # Train
        fold_model.train()
        train_losses = []
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            optimiser.zero_grad()
            preds = fold_model(imgs)
            loss = criterion(preds, targets)
            loss.backward()
            optimiser.step()
            train_losses.append(loss.item())

        # Validate
        fold_model.eval()
        val_losses = []
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                preds = fold_model(imgs)
                val_losses.append(criterion(preds, targets).item())

        val_loss = np.mean(val_losses)
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:02d} | Train loss: {np.mean(train_losses):.4f} | Val loss: {val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in fold_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # Load best weights
    fold_model.load_state_dict(best_state)

    # Predict on val — store per-image predictions
    fold_model.eval()
    val_img_preds = []
    val_img_lsoas = val_df['lsoa'].tolist()
    val_img_imds  = val_df['imd'].tolist()

    with torch.no_grad():
        for imgs, _ in val_loader:
            preds = fold_model(imgs.to(device))
            val_img_preds.extend((preds.cpu().numpy() * y_std + y_mean).tolist())

    # Aggregate predictions per LSOA (prediction averaging — professor's suggestion)
    lsoa_pred_map = {}
    for lsoa, pred, imd in zip(val_img_lsoas, val_img_preds, val_img_imds):
        if lsoa not in lsoa_pred_map:
            lsoa_pred_map[lsoa] = {'preds': [], 'true': imd}
        lsoa_pred_map[lsoa]['preds'].append(pred)

    lsoa_preds_agg = [np.mean(v['preds']) for v in lsoa_pred_map.values()]
    lsoa_true_agg  = [v['true'] for v in lsoa_pred_map.values()]

    fold_r2 = r2_score(lsoa_true_agg, lsoa_preds_agg)
    all_fold_r2.append(fold_r2)
    print(f'  Fold {fold+1} LSOA-level R² (prediction averaging): {fold_r2:.4f}')

    oof_lsoa_preds.update({l: np.mean(v['preds']) for l, v in lsoa_pred_map.items()})
    oof_lsoa_true.update({l: v['true'] for l, v in lsoa_pred_map.items()})

print(f'\n=== Fine-tuned Model Results ===')
print(f'Mean fold R²: {np.mean(all_fold_r2):.4f} ± {np.std(all_fold_r2):.4f}')

all_preds = [oof_lsoa_preds[l] for l in oof_lsoa_preds]
all_true  = [oof_lsoa_true[l]  for l in oof_lsoa_preds]
final_r2  = r2_score(all_true, all_preds)
print(f'Overall OOF R² (images only, spatial CV): {final_r2:.4f}')


=== Fold 1/5 | Test boroughs: 11 ===
Train images: 5616 | Val images: 1308


In [ ]:
# ── Plot results ──────────────────────────────────────────────────────────────
os.makedirs('outputs', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
ax.scatter(all_true, all_preds, alpha=0.5, s=25, color='steelblue', edgecolors='none')
lims = [min(all_true), max(all_true)]
ax.plot(lims, lims, 'r--', lw=1.5)
ax.set_title(f'Fine-tuned ResNet-50 (Spatial CV)\nR² = {final_r2:.4f}', fontsize=12)
ax.set_xlabel('True IMD Rank')
ax.set_ylabel('Predicted IMD Rank')

# Fold R²
ax2 = axes[1]
ax2.bar(range(1, 6), all_fold_r2, color='steelblue')
ax2.axhline(np.mean(all_fold_r2), color='red', linestyle='--', label=f'Mean={np.mean(all_fold_r2):.3f}')
ax2.axhline(0.7, color='green', linestyle=':', label='Target R²=0.70')
ax2.set_xlabel('Fold')
ax2.set_ylabel('R²')
ax2.set_title('R² per Spatial Fold (Fine-tuned)')
ax2.legend()
ax2.set_ylim(0, 1.0)

plt.suptitle('Partial Fine-tuning Results — Borough Holdout CV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/finetune_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSummary:')
print(f'  Frozen baseline R²:      0.2666  # from notebook3_main spatial CV')
print(f'  Fine-tuned R²:           {final_r2:.4f}')
print(f'  Improvement:             +{final_r2 - 0.2666:.4f}')

In [ ]:
# ── Save final model (retrain on all data) ────────────────────────────────────
os.makedirs('models', exist_ok=True)

print('Training final model on all data...')
y_mean_all = img_df['imd'].mean()
y_std_all  = img_df['imd'].std()

full_ds = StreetViewDataset(img_df, train_transform, y_mean_all, y_std_all)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=True, num_workers=0)

final_model = PartialFineTuneResNet(dropout=0.4).to(device)
optimiser_f = optim.AdamW([
    {'params': list(final_model.backbone.parameters()), 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': list(final_model.head.parameters()),     'lr': 1e-3, 'weight_decay': 1e-3}
])
scheduler_f = optim.lr_scheduler.CosineAnnealingLR(optimiser_f, T_max=20)
criterion_f = nn.MSELoss()

for epoch in range(20):
    final_model.train()
    losses = []
    for imgs, targets in full_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimiser_f.zero_grad()
        loss = criterion_f(final_model(imgs), targets)
        loss.backward()
        optimiser_f.step()
        losses.append(loss.item())
    scheduler_f.step()
    if (epoch+1) % 5 == 0:
        print(f'  Epoch {epoch+1:02d} | Loss: {np.mean(losses):.4f}')

torch.save({
    'model_state': final_model.state_dict(),
    'y_mean': y_mean_all,
    'y_std': y_std_all
}, 'models/finetune_final.pt')

print('Saved to models/finetune_final.pt')